In [4]:
import pandas as pd
import numpy as np

### Funciones

In [5]:
def calcular_error_medicion(escala, vpp):
    """
    Calcula el error de medición en función de la escala de medición.

    Parámetros:
    escala (pd.Series): Serie con las escalas de medición.
    vpp (pd.Series): Serie con los valores de Vpp.

    Retorna:
    pd.Series: Serie con los errores de medición calculados.
    """

    # 1. Definimos las condiciones (en Voltios)
    condiciones = [
        (escala >= 0.002) & (escala <= 0.005),  # 2mV a 5mV -> 4%
        (escala >= 0.010) & (escala <= 10.0),  # 10mV a 10V -> 3%
    ]
    # 2. Definimos los porcentajes correspondientes
    porcentajes = [0.04, 0.03]
    # 3. Asignamos el porcentaje según la escala
    pct_aplicable = np.select(condiciones, porcentajes, default=np.nan)
    # 4. Calculamos el error de medición
    error_vpp = vpp * pct_aplicable
    return error_vpp


In [29]:
def redondear_cifras_significativas(valor, n):
    """
    Redondea un valor a un número específico de cifras significativas.
    """
    if pd.isna(valor) or valor == 0:
        return valor
    # Calcula los decimales necesarios según la magnitud del número
    decimales = n - 1 - int(np.floor(np.log10(abs(valor))))
    return round(valor, decimales)

In [33]:
def redondear_medicion(row, valor_verdadero, incerteza):
    """
    Redondea una medición según su incertidumbre.
    """
    medicion = row[valor_verdadero]
    incertidumbre = row[incerteza]

    if pd.isna(medicion) or pd.isna(incertidumbre) or incertidumbre == 0:
        return medicion

    # Determina a qué posición decimal debemos redondear
    # Ejemplo: para 0.3 -> -log10(0.3) = 0.52 -> floor = 0 -> decimales = 1
    # Ejemplo: para 0.04 -> -log10(0.04) = 1.39 -> floor = 1 -> decimales = 2
    decimales = int(-np.floor(np.log10(abs(incertidumbre))))

    if decimales >= 0:
        return round(medicion, decimales)
    else:
        # Si la incertidumbre está en las decenas/centenas (ej: ±30)
        return round(medicion, decimales)

### Importamos los datos

In [2]:
path = r".\clase1\raw\mediciones1_raw.csv"

In [18]:
df = pd.read_csv(path, sep=";")
df.head(5)

,Vpp (V),Vt (V) a 10 Hz,Vt (V) a 100 Hz,Vt (V) a 1 KHz,Vt (V) a 10 KHz,Vt (V) a 100 KHz,Vt (V) a 1 MHz,Escala (V/d),Unnamed: 8
0,2.04,0.43,0.63,0.60,0.43,0.02,0.010,0.5,NaN
1,4.04,1.25,1.34,1.28,1.54,0.24,0.003,1.0,NaN
2,6.00,2.04,2.11,2.01,2.73,1.06,0.003,1.0,NaN
3,8.00,2.82,2.79,2.68,3.91,1.61,0.002,2.0,NaN
4,10.00,3.58,3.54,3.43,5.03,1.87,0.002,2.0,NaN


In [19]:
df.shape

(11, 9)

In [21]:
df = df.drop(columns=["Unnamed: 8"])

In [26]:
df = df.dropna()

### Limpiando datos

In [34]:
df_clean = pd.DataFrame()
df_clean["vpp"] = df["Vpp (V)"]
df_clean["vpp_error"] = calcular_error_medicion(df["Escala (V/d)"], df["Vpp (V)"])
df_clean["vpp_error"] = df_clean["vpp_error"].apply(lambda x: redondear_cifras_significativas(x, 1))
df_clean["vpp"] = df_clean.apply(lambda row: redondear_medicion(row, "vpp", "vpp_error"), axis=1)
df_clean

,vpp,vpp_error
0,2.04,0.06
1,4.00,0.10
2,6.00,0.20
3,8.00,0.20
4,10.00,0.30
5,12.00,0.40
6,14.10,0.40
7,16.00,0.50
8,18.00,0.50
9,20.00,0.60


In [37]:
vt_error = np.array([0.01]*10)
vt_error

array([0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01])

In [38]:
name_columns = ["10 Hz", "100 Hz", "1 KHz", "10 KHz", "100 KHz", "1 MHz"]

In [45]:
for i, name in enumerate(name_columns, start=1):
    df_clean[name] = df.iloc[:, i:i+1]

In [47]:
df_clean["vt_error"] = vt_error

In [48]:
df_clean

,vpp,vpp_error,10 Hz,100 Hz,1 KHz,10 KHz,100 KHz,1 MHz,vt_error
0,2.04,0.06,0.43,0.63,0.60,0.43,0.02,0.010,0.01
1,4.00,0.10,1.25,1.34,1.28,1.54,0.24,0.003,0.01
2,6.00,0.20,2.04,2.11,2.01,2.73,1.06,0.003,0.01
3,8.00,0.20,2.82,2.79,2.68,3.91,1.61,0.002,0.01
4,10.00,0.30,3.58,3.54,3.43,5.03,1.87,0.002,0.01
5,12.00,0.40,4.34,4.24,4.13,6.24,1.96,0.002,0.01
6,14.10,0.40,5.15,4.98,4.87,7.42,2.01,0.002,0.01
7,16.00,0.50,5.87,5.53,5.42,8.32,2.04,0.002,0.01
8,18.00,0.50,6.47,6.24,6.09,9.58,2.06,0.001,0.01
9,20.00,0.60,7.18,6.97,6.85,10.83,2.07,0.001,0.01


### Guardando los datos

In [49]:
path_clean = r".\clase1\clean\mediciones1_clean.csv"

In [50]:
df_clean.to_csv(path_clean, index=False)